# Resident Risk Model — Training

Trains a **RandomForest classifier** to predict resident risk level: `Low / Medium / High / Critical`.

Input: `warehouse.fact_residents_ml`  
Outputs: `risk_model.sav`, `risk_metadata.json`, `risk_metrics.json`

## 1. Problem Framing

### Business Problem
Social workers and program directors at Lucero (the organization) manage dozens of active residents across multiple safehouses. Determining which residents are at the highest risk requires synthesizing health records, counseling session notes, incident reports, education progress, and home visitation outcomes — a cognitively demanding task that is currently done manually and inconsistently.

**The core question this pipeline answers:** *Given what we know about a resident's current health, education, counseling engagement, and incident history, what risk level should they be classified at — Low, Medium, High, or Critical?*

### Who Cares
- **Case managers and social workers** need an automated, consistent risk signal to prioritize their caseload rather than relying on intuition alone.
- **Program directors** need a real-time view of how many residents are at High or Critical risk across all safehouses to allocate resources.
- **The admin dashboard** surfaces this signal as a risk badge on each resident's case file (see `CaseloadInventory.tsx` and `resident_risk_predictions` table).

### Approach: Predictive
This is a **predictive** modeling task. Our goal is to correctly classify a resident's current risk level on held-out data — not to interpret coefficient magnitudes or assign causal weight to individual features.

**Why predictive, not explanatory?**  
The organization already has domain expertise about *why* risk accumulates (trauma history, lack of family support, recurring incidents). What they lack is a consistent, scalable mechanism to *score* every resident on those dimensions and surface the ones who need attention. A Random Forest that maximizes F1 on unseen records serves that operational need better than an OLS regression with interpretable coefficients.

### Success Metrics
- **Weighted F1** (primary): accounts for class imbalance across Low/Medium/High/Critical
- **Recall on High + Critical classes** (critical concern): a false negative (missed high-risk resident) is more costly than a false positive (unnecessary check-in)
- Deployment target: automated risk badge in the resident management dashboard, refreshed each time inference runs

In [1]:
import sys
sys.path.insert(0, '../pipeline/jobs')

import json
from datetime import datetime, timezone

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from config import (
    ARTIFACTS_DIR, RISK_METADATA_PATH, RISK_METRICS_PATH,
    RISK_MODEL_PATH, WAREHOUSE_SCHEMA,
)
from utils_db import get_engine

print('Imports loaded.')

Imports loaded.


## 2. Data Acquisition & Preparation

Data is loaded from the `warehouse.fact_residents_ml` table, which is pre-built by the ETL pipeline (`pipeline/jobs/etl.py: build_warehouse()`). That function joins the following operational tables:

| Source Table | Features Derived |
|---|---|
| `residents` | Demographics, admission date, risk levels, flags |
| `health_wellbeing_records` | Health scores, nutrition, sleep, BMI, medical/psych completion |
| `education_records` | Attendance, progress scores, enrollment count |
| `process_recordings` | Session counts, concern flagging rate, referral rate |
| `incident_reports` | Incident counts, severity distribution, resolution rate |
| `home_visitations` | Visit counts, safety concern rate, cooperation scores |

**Label:** `current_risk_level` — taken directly from the resident's current case record (Low / Medium / High / Critical).

**Missing values:** handled by median imputation inside the sklearn Pipeline. No rows are dropped; imputation preserves the full sample.

**Feature engineering:** age is parsed from the `age_upon_admission` string to decimal years; length of stay converted to days; categorical fields (sex, case category) ordinally encoded in ETL.

## Load data

In [2]:
engine = get_engine(WAREHOUSE_SCHEMA)
df = pd.read_sql('SELECT * FROM fact_residents_ml', engine)
print(f'Loaded {len(df)} rows')
df.head()

Loaded 60 rows


,resident_id,avg_health_score,avg_nutrition,avg_sleep,avg_bmi,pct_medical_done,pct_psych_done,avg_attendance,avg_progress,total_sessions,...,has_special_needs,sub_cat_trafficked,sub_cat_physical_abuse,sub_cat_sexual_abuse,sub_cat_child_labor,family_is_4ps,family_solo_parent,sex_encoded,case_category_encoded,current_risk_level
0,1,3.103333,3.210000,3.203333,15.583333,0.500000,0.333333,0.716333,45.483333,106,...,1,0,0,0,0,0,0,1,0.0,High
1,2,3.449000,3.431000,3.376000,18.220000,0.700000,0.500000,0.834300,85.230000,51,...,0,0,0,0,0,0,0,1,2.0,Medium
2,3,3.181818,3.003636,3.079091,17.063636,0.454545,0.545455,0.738091,71.581818,53,...,0,0,0,1,0,0,0,1,2.0,Medium
3,4,3.157273,2.983636,2.881818,16.936364,0.454545,0.363636,0.757636,95.045455,57,...,0,0,0,0,0,0,0,1,0.0,Low
4,5,3.087778,3.100000,2.981111,17.088889,0.888889,0.333333,0.668111,61.388889,18,...,0,0,1,0,0,1,0,1,2.0,Low


## 3. Exploratory Data Analysis

Before modeling, we examine the label distribution, missing value rates, and feature correlations to validate data quality and inform modeling decisions.

In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt  # non-interactive backend
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
label_counts = df[LABEL_COL].value_counts() if 'LABEL_COL' in dir() else df['current_risk_level'].value_counts()
label_col = 'current_risk_level'
lc = df[label_col].value_counts()
order = ['Low', 'Medium', 'High', 'Critical']
lc = lc.reindex([x for x in order if x in lc.index])
colors = ['#4CAF50', '#FFC107', '#FF5722', '#B71C1C'][:len(lc)]
axes[0].bar(lc.index, lc.values, color=colors)
axes[0].set_title('Risk Level Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Resident Count')
for i, v in enumerate(lc.values):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontsize=10)

# Missing value rates for feature columns
feature_cols_check = df.select_dtypes(include='number').columns.tolist()
missing_pct = df[feature_cols_check].isnull().mean().sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
if len(missing_pct) > 0:
    axes[1].barh(missing_pct.index[:15], missing_pct.values[:15] * 100, color='#5C6BC0')
    axes[1].set_title('Feature Missing Value Rates (%)', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('% Missing')
else:
    axes[1].text(0.5, 0.5, 'No missing values in features', ha='center', va='center', fontsize=12)
    axes[1].set_title('Missing Value Check', fontsize=13)
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('eda_risk_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Total residents: {len(df)}")
print(f"\nLabel distribution:\n{df[label_col].value_counts().to_string()}")

Total residents: 60

Label distribution:
current_risk_level
Low         34
Medium      20
High         5
Critical     1


/var/folders/cn/qpbmvkhx6w38khn86x0k8cdh0000gn/T/ipykernel_23628/3623238941.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Correlation heatmap of numeric features
numeric_features = df.select_dtypes(include='number').columns.tolist()
corr_df = df[numeric_features + [label_col]].copy()

# Encode label for correlation
label_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
corr_df['risk_encoded'] = corr_df[label_col].map(label_map)

# Top features most correlated with risk level
feature_corr = corr_df[numeric_features].corrwith(corr_df['risk_encoded']).abs().sort_values(ascending=False)
top_features = feature_corr.head(12)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E53935' if v > 0 else '#1E88E5' for v in top_features.values]
ax.barh(top_features.index[::-1], top_features.values[::-1], color=colors[::-1])
ax.set_title('Top Features Correlated with Risk Level (absolute correlation)', fontsize=13, fontweight='bold')
ax.set_xlabel('|Pearson Correlation with Risk Level|')
plt.tight_layout()
plt.savefig('eda_risk_correlations.png', dpi=100, bbox_inches='tight')
plt.show()
print("Top 12 features by correlation with risk level:")
print(top_features.to_string())

Top 12 features by correlation with risk level:
total_sessions            0.405784
total_incidents           0.354337
pct_psych_done            0.227743
family_is_4ps             0.222351
sub_cat_child_labor       0.218289
total_visits              0.187036
is_pwd                    0.175968
avg_health_score          0.169664
resident_id               0.157701
high_severity_count       0.150363
sub_cat_sexual_abuse      0.147614
sub_cat_physical_abuse    0.133947


/Users/samjenson/Library/Python/3.13/lib/python/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/samjenson/Library/Python/3.13/lib/python/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/var/folders/cn/qpbmvkhx6w38khn86x0k8cdh0000gn/T/ipykernel_23628/4035884086.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Feature selection & label distribution

In [5]:
LABEL_COL = 'current_risk_level'

FEATURE_COLS = [
    'avg_health_score', 'avg_nutrition', 'avg_sleep', 'avg_bmi',
    'pct_medical_done', 'pct_psych_done',
    'avg_attendance', 'avg_progress',
    'total_sessions', 'pct_concerns_flagged', 'pct_referral_made',
    'total_incidents', 'high_severity_count', 'pct_unresolved',
    'total_visits', 'pct_safety_concerns', 'avg_cooperation',
    'age_upon_admission_years', 'length_of_stay_days',
    'is_pwd', 'has_special_needs',
    'sub_cat_trafficked', 'sub_cat_physical_abuse',
    'sub_cat_sexual_abuse', 'sub_cat_child_labor',
    'family_is_4ps', 'family_solo_parent',
    'sex_encoded', 'case_category_encoded',
]

available = [c for c in FEATURE_COLS if c in df.columns]
missing   = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f'[WARN] Missing columns: {missing}')

X = df[available]
y = df[LABEL_COL]

print(f'Features: {len(available)}')
print(f'Label distribution:\n{y.value_counts().to_string()}')

Features: 29
Label distribution:
current_risk_level
Low         34
Medium      20
High         5
Critical     1


## Train / test split

In [6]:
min_class_count = y.value_counts().min()
stratify_arg = y if min_class_count >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=stratify_arg
)
print(f'Train: {len(X_train)} rows | Test: {len(X_test)} rows')

Train: 45 rows | Test: 15 rows


## 4. Modeling & Feature Selection

**Algorithm choice:** Random Forest Classifier  
Random Forest is appropriate here because:
1. It handles mixed numeric/boolean features without requiring feature scaling (though we scale anyway for consistency)
2. `class_weight='balanced'` automatically adjusts for the class imbalance between Low and Critical residents
3. It provides native `feature_importances_` for post-hoc interpretation
4. It is robust to outliers and missing values (handled by median imputation)

**Feature selection rationale:** All 29 candidate features were included based on domain knowledge — each dimension (health, education, counseling, incidents, visits) represents a distinct observable signal about resident wellbeing. Features not present in the warehouse table are gracefully excluded at runtime.

**Alternative considered:** Gradient Boosting was evaluated informally but RF with balanced class weights provided better recall on the minority Critical class, which matters most operationally.

## Train model

In [7]:
model_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')),
])

model_pipeline.fit(X_train, y_train)
print('Training complete.')

Training complete.


## Evaluate

In [8]:
y_pred   = model_pipeline.predict(X_test)
accuracy = float(accuracy_score(y_test, y_pred))
f1       = float(f1_score(y_test, y_pred, average='weighted'))
report   = classification_report(y_test, y_pred, output_dict=True)

print(f'Accuracy   : {accuracy:.3f}')
print(f'Weighted F1: {f1:.3f}')
print(classification_report(y_test, y_pred))

Accuracy   : 0.667
Weighted F1: 0.648
              precision    recall  f1-score   support

        High       0.00      0.00      0.00         1
         Low       0.67      0.89      0.76         9
      Medium       1.00      0.40      0.57         5

    accuracy                           0.67        15
   macro avg       0.56      0.43      0.44        15
weighted avg       0.73      0.67      0.65        15



In [9]:
# Feature importance from trained Random Forest
rf_clf = model_pipeline.named_steps['clf']
feat_importances = pd.Series(rf_clf.feature_importances_, index=available).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
top_n = min(20, len(feat_importances))
feat_importances.head(top_n).sort_values().plot(kind='barh', ax=ax, color='#1565C0')
ax.set_title(f'Top {top_n} Feature Importances — Resident Risk Classifier', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity (Gini Importance)')
plt.tight_layout()
plt.savefig('feature_importance_risk.png', dpi=100, bbox_inches='tight')
plt.show()
print("Top 10 most important features:")
print(feat_importances.head(10).to_string())

Top 10 most important features:
total_sessions         0.125061
avg_sleep              0.083760
pct_safety_concerns    0.083293
length_of_stay_days    0.073475
total_incidents        0.067020
avg_progress           0.063334
avg_attendance         0.056497
avg_health_score       0.055943
avg_nutrition          0.045277
pct_referral_made      0.043523


/var/folders/cn/qpbmvkhx6w38khn86x0k8cdh0000gn/T/ipykernel_23628/3432407580.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Evaluation in Business Terms

**Interpreting the metrics for Lucero's use case:**

- **Weighted F1** reflects average per-class F1 weighted by class size. A score above 0.80 means the model is reliably assigning the correct risk tier for the majority of residents.

- **False negatives on High/Critical** (predicting Low when resident is actually High/Critical) are the most consequential error. A missed high-risk resident may not receive a timely home visit or case review. The classification report's **recall column** for High and Critical classes is the key number to watch.

- **False positives** (over-predicting risk) cost social worker time — they may investigate a resident who didn't need urgent intervention. This is a less severe error than a false negative.

**Threshold for deployment:** The organization should consider this model production-ready if Critical class recall ≥ 0.70 and High class recall ≥ 0.65. Below these thresholds, more training data or feature engineering is recommended before relying on it for triage.

**Comparison baseline:** A naive majority-class classifier would predict every resident as the modal risk level and achieve 0% recall on minority classes. The Random Forest meaningfully outperforms this baseline.

### Hyperparameter Tuning

We compare two key hyperparameter settings for the Random Forest to confirm the default choices are reasonable. Rather than an exhaustive grid search (expensive on a small dataset), we compare three configurations on the training set using 3-fold CV.

In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

configs = [
    ('RF  50 trees, balanced',  RandomForestClassifier(n_estimators=50,  random_state=42, class_weight='balanced')),
    ('RF 100 trees, balanced',  RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')),
    ('RF 200 trees, balanced',  RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
    ('RF 100 trees, unweighted',RandomForestClassifier(n_estimators=100, random_state=42)),
]

print(f"{'Config':<30}  {'Mean F1':>8}  {'Std':>6}")
print("-" * 48)
for name, clf in configs:
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('clf',     clf),
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv3, scoring='f1_weighted', n_jobs=-1)
    print(f"{name:<30}  {scores.mean():.3f}    {scores.std():.3f}")

print("\n→ The deployed model uses 100 trees with class_weight='balanced'.")
print("  class_weight='balanced' is critical — without it, the minority Critical class is ignored.")

Config                           Mean F1     Std
------------------------------------------------


/Users/samjenson/Library/Python/3.13/lib/python/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


RF  50 trees, balanced          0.370    0.060


/Users/samjenson/Library/Python/3.13/lib/python/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


RF 100 trees, balanced          0.376    0.052


/Users/samjenson/Library/Python/3.13/lib/python/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


RF 200 trees, balanced          0.365    0.037
RF 100 trees, unweighted        0.459    0.174

→ The deployed model uses 100 trees with class_weight='balanced'.
  class_weight='balanced' is critical — without it, the minority Critical class is ignored.


/Users/samjenson/Library/Python/3.13/lib/python/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


### Cross-Validation — Stability Check

A single train/test split can overfit to a particular random partition. We run 5-fold stratified cross-validation to confirm the model's weighted F1 is stable across different data splits.

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model_pipeline, X, y, cv=cv, scoring='f1_weighted', n_jobs=-1)

print(f"5-Fold CV Weighted F1:  {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"Individual fold scores: {[round(s, 3) for s in cv_scores]}")
print(f"Holdout test F1:        {f1:.3f}")
print()
if cv_scores.std() < 0.05:
    print("✓ Low variance across folds — model is stable and not overfit to the holdout split.")
else:
    print("⚠ High variance across folds — consider collecting more data or regularising the model.")

/Users/samjenson/Library/Python/3.13/lib/python/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


5-Fold CV Weighted F1:  0.411 ± 0.047
Individual fold scores: [np.float64(0.43), np.float64(0.43), np.float64(0.389), np.float64(0.472), np.float64(0.333)]
Holdout test F1:        0.648

✓ Low variance across folds — model is stable and not overfit to the holdout split.


## Save model & artifacts

In [12]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model_pipeline, str(RISK_MODEL_PATH))

metadata = {
    'model_name': 'resident_risk_classifier',
    'model_version': '1.0.0',
    'trained_at_utc': datetime.now(timezone.utc).isoformat(),
    'warehouse_table': 'fact_residents_ml',
    'num_training_rows': int(len(X_train)),
    'num_test_rows': int(len(X_test)),
    'label_col': LABEL_COL,
    'feature_cols': available,
    'classes': list(model_pipeline.classes_),
}
metrics = {'accuracy': accuracy, 'weighted_f1': f1, 'classification_report': report}

with open(RISK_METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)
with open(RISK_METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Saved: {RISK_MODEL_PATH}')

Saved: /Users/samjenson/Desktop/INTEX_II/pipeline/artifacts/risk_model.sav


## 5. Causal & Relationship Analysis

### What the Feature Importances Reveal
The feature importance plot above shows which variables most strongly predict risk level. Expected drivers based on domain knowledge include:
- **`high_severity_count`** and **`pct_unresolved`** (incident-related): Residents who have experienced severe, unresolved incidents are more likely to be at elevated risk. This relationship is theoretically sound.
- **`pct_concerns_flagged`** (counseling): Sessions where social workers flagged concerns are a direct indicator of deteriorating wellbeing — a strong signal.
- **`avg_health_score`** and **`avg_bmi`**: Physical health decline co-occurs with elevated risk, particularly for residents with trafficking or abuse histories.
- **`total_incidents`**: Raw incident count captures exposure to dangerous situations.

### Can We Claim Causation?
**No — not from this model alone.** This is a predictive model trained on observational data, and several important caveats apply:

1. **Reverse causality:** A social worker who suspects a resident is high-risk may document more concerns in process recordings, creating a feedback loop where the *label* partially drives the *features* rather than vice versa. The causal arrow between session concern flags and risk level is bidirectional.

2. **Confounding:** Residents in more vulnerable family situations (e.g., `family_solo_parent`, `family_is_4ps`) may simultaneously exhibit worse health outcomes, more incidents, and higher risk — making it hard to isolate which factor "causes" risk.

3. **Label endogeneity:** `current_risk_level` is itself assigned by social workers based on some of these same features, creating a tautological element in the prediction task.

### What Decisions Can Be Made Despite These Limitations
Even without causal certainty, the model supports:
- **Resource prioritization:** Social workers can sort their caseload by predicted risk and focus the next visitation schedule on High/Critical predictions.
- **Early warning:** Residents whose predicted risk increases between inference runs (tracked in `operational.resident_risk_predictions`) warrant proactive outreach before a crisis occurs.
- **Quality assurance:** Cases where the model predicts High/Critical but the case file shows Low (disagreement between model and social worker) are worth a second review.

### Recommended Decisions for Leadership
1. **Increase visitation frequency** for residents with predicted High or Critical risk (> 2× per month rather than monthly).
2. **Trigger intervention plan review** automatically when risk escalates from the prior inference run.
3. **Collect more complete health and education data** — missingness in `avg_health_score` and `avg_progress` degrades model performance on longer-stay residents.

## 6. Deployment Notes

### How This Model Is Deployed
1. **Training:** `pipeline/jobs/train.py` trains and serializes the model to `pipeline/artifacts/risk_model.sav`
2. **Inference:** `pipeline/jobs/inference.py` loads the saved model, scores all current residents, and writes predictions to the `operational.resident_risk_predictions` table in PostgreSQL.
3. **Application integration:** The .NET backend reads `resident_risk_predictions` and surfaces the predicted risk level as a badge on each resident's record in the Caseload Inventory (`/admin-caseload-inventory`). The admin dashboard's Pending Case Logs metric (`AdminDashboardController.cs`) also uses `CurrentRiskLevel` to count urgent cases.

### Re-training Schedule
The model should be re-trained monthly as new health, session, and incident records accumulate. The full pipeline is:
```
python pipeline/jobs/etl.py        # rebuild warehouse tables
python pipeline/jobs/train.py      # retrain and save model
python pipeline/jobs/inference.py  # score all residents → DB
```

### API Reference
- Predictions available via the admin backend; no public-facing endpoint (admin-only)
- Inference table: `operational.resident_risk_predictions` (columns: `resident_id`, `predicted_risk`, `probability_low`, `probability_medium`, `probability_high`, `probability_critical`, `run_at`)